# 10 — Action Space Sanity Check

Thin orchestration notebook (Task 9). Builds the price-space levels
(`azlib.space.price_levels`) and the `label_coeff` regression target
(`azlib.space.label_coeff`) for one `(tf, direction)`, then sanity-plots
both. All math lives in `azlib.space` (Task 2, tested) — this notebook only
loads data, calls it, and plots/reports the result.

Dataset comes from the ambient environment (`ROOT_FOLDER`/`DATA_ROOT`/
`DATA_SET_NAME`/`PAIR` — set via `docker compose run --env-file ...`, see
`nb/README.md`), matching `azlib.loader.load_wide_df()`'s own env-driven
convention.

In [ ]:
tf = 15
direction = "long"

In [ ]:
import sys
from pathlib import Path

# Bootstrap: notebooks/action_zones/{azlib,nb} onto sys.path. Assumes cwd is
# the repo root (/code) -- true for the experiment Docker service
# (docker-compose.experiment.yml's working_dir: /code; papermill's own
# cwd=None default inherits the invoking process's cwd) and for the
# automated smoke test (tests/test_layer9_notebooks.py sets the papermill
# subprocess's cwd to the repo root to match). helpers.py/indicators.*
# resolve for free via the image's own PYTHONPATH=/code.
_AZ_ROOT = Path.cwd() / "notebooks" / "action_zones"
for _p in (_AZ_ROOT, _AZ_ROOT / "nb"):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

import nbutil

In [ ]:
from azlib.loader import load_wide_df
from azlib.space import price_levels, label_coeff

wide_df = load_wide_df()
print(f"wide_df: {len(wide_df)} rows, tf={tf}, direction={direction}")

In [ ]:
price_high_level, price_low_level = price_levels(wide_df, tf)
lc = label_coeff(wide_df, tf, direction)

n_total = len(wide_df)
n_level_nonnan = int(price_high_level.notna().sum())
n_coeff_nonnan = int(lc.notna().sum())
print(f"price_levels non-NaN: {n_level_nonnan}/{n_total}")
print(f"label_coeff non-NaN: {n_coeff_nonnan}/{n_total}")

In [ ]:
import matplotlib
matplotlib.use("Agg", force=True)
import matplotlib.pyplot as plt

out_dir = nbutil.results_dir(tf, direction, "space")

entry_col = "1_low" if direction == "long" else "1_high"
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(wide_df.index, wide_df[entry_col], label=entry_col, linewidth=0.7, color="black")
ax.plot(wide_df.index, price_high_level, label="price_high_level", linewidth=0.9, color="tab:green")
ax.plot(wide_df.index, price_low_level, label="price_low_level", linewidth=0.9, color="tab:red")
ax.set_title(f"Price space sanity — tf={tf} direction={direction}")
ax.legend(loc="best")
fig.tight_layout()
price_levels_path = f"{out_dir}/price_levels.png"
fig.savefig(price_levels_path)
plt.close(fig)
print(price_levels_path)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(lc.dropna(), bins=40, color="tab:blue", alpha=0.8)
ax.axvline(0.5, color="black", linestyle="--", linewidth=1)
ax.set_title(f"label_coeff distribution — tf={tf} direction={direction}")
ax.set_xlabel("label_coeff")
fig.tight_layout()
label_coeff_hist_path = f"{out_dir}/label_coeff_hist.png"
fig.savefig(label_coeff_hist_path)
plt.close(fig)
print(label_coeff_hist_path)

In [ ]:
lc_nonnan = lc.dropna()
report = {
    "tf": tf,
    "direction": direction,
    "n_rows": n_total,
    "price_levels_non_nan": n_level_nonnan,
    "label_coeff_non_nan": n_coeff_nonnan,
    "label_coeff_mean": float(lc_nonnan.mean()) if n_coeff_nonnan else None,
    "label_coeff_std": float(lc_nonnan.std()) if n_coeff_nonnan else None,
    "label_coeff_min": float(lc_nonnan.min()) if n_coeff_nonnan else None,
    "label_coeff_max": float(lc_nonnan.max()) if n_coeff_nonnan else None,
    "frac_coeff_ge_half": float((lc_nonnan >= 0.5).mean()) if n_coeff_nonnan else None,
    "price_levels_png": price_levels_path,
    "label_coeff_hist_png": label_coeff_hist_path,
}
report_path = nbutil.write_json_report(f"{out_dir}/space_report.json", report)
print(report_path)
report